In [1]:
import os
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

In [2]:
data_path = Path(os.environ["DATA_PATH"])
initial_path = data_path / "initial"
generated_path = data_path / "generated"

population_grids_path = Path(os.environ["POPULATION_GRIDS_PATH"])

# Geometría

In [3]:
df_geom = (
    gpd.read_file(initial_path / "r_02a_zonas", columns=["CVEGEO", "_zonaurban"])
    .rename(columns={"_zonaurban": "zone"})
    .query("zone.notna()")
    .assign(zone=lambda df: df["zone"].astype(int).sub(1).astype("category"))
    .set_index("CVEGEO")
    .to_crs("EPSG:6372")
)

# Ingreso

In [5]:
income = gpd.read_file(initial_path / "income.gpkg").set_index("cvegeo")["income_pc"]

# Área baldía

In [6]:
df_barren = (
    gpd.read_file(initial_path / "baldios")
    .assign(geometry=lambda df: df["geometry"].force_2d())
    .to_crs("EPSG:6372")
)
barren_area_frac = (
    df_geom.reset_index()
    .overlay(df_barren, how="intersection")
    .assign(area=lambda df: df["geometry"].area)
    .groupby("CVEGEO")["area"]
    .sum()
    .div(df_geom["geometry"].area)
)

# Delitos

In [7]:
crimes = gpd.read_file(initial_path / "num_delitos_ageb").set_index("CVEGEO")[
    "num_delito"
]

# Valor catastral

In [8]:
df_cat = (
    gpd.read_file(
        initial_path / "valor_cat_actualizado",
        columns=["secsub", "valor2024"],
    )
    .dropna(subset=["secsub"])
    .drop(columns=["secsub"])
    .assign(geometry=lambda df: df["geometry"].force_2d())
    .to_crs("EPSG:6372")
)

# Censo

In [9]:
wanted_cols = [
    "POBTOT",
    "P_60YMAS",
    "P_0A2",
    "P_3A5",
    "P_6A11",
    "P_12A14",
    "P_15A17",
    "P_18A24",
    "POB15_64",
    "TVIVPAR",
    "TVIVPARHAB",
    "GRAPROES",
]

df_census = (
    pd.read_csv(
        initial_path / "conjunto_de_datos_ageb_urbana_02_cpv2020.csv",
        usecols=[*wanted_cols, "ENTIDAD", "MUN", "LOC", "AGEB", "MZA"],
    )
    .query("MZA == 0")
    .assign(
        CVEGEO=lambda df: df["ENTIDAD"].astype(str).str.zfill(2)
        + df["MUN"].astype(str).str.zfill(3)
        + df["LOC"].astype(str).str.zfill(4)
        + df["AGEB"].astype(str).str.zfill(4),
    )
    .drop(
        columns=[
            "ENTIDAD",
            "MUN",
            "LOC",
            "AGEB",
            "MZA",
        ],
    )
    .set_index("CVEGEO")[wanted_cols]
    .replace("*", np.nan)
)

for c in df_census.columns:
    if c.startswith("P") and df_census[c].dtype == "str":
        try:
            df_census[c] = df_census[c].astype(int)
        except ValueError:
            df_census[c] = df_census[c].astype(float)

# Final

In [10]:
df_agebs = (
    df_geom.join(df_census, how="inner")
    .assign(
        income=income,
        vacant_area_frac=barren_area_frac,
        crimes=crimes,
        area_km2=lambda df: df["geometry"].area.div(1e6),
    )
    .sort_index()
)
df_agebs.to_file(generated_path / "agebs.gpkg")